# PepperGPT

In [26]:
# built-in imports
!pip install requests
!pip install speechrecognition==3.8.1
!pip install paramiko
!pip install pandas

DEPRECATION: Python 2.7 reached the end of its life on January 1st, 2020. Please upgrade your Python as Python 2.7 is no longer maintained. A future version of pip will drop support for Python 2.7. More details about Python 2 support in pip, can be found at https://pip.pypa.io/en/latest/development/release-process/#python-2-support
You should consider upgrading via the '/usr/local/bin/python -m pip install --upgrade pip' command.
DEPRECATION: Python 2.7 reached the end of its life on January 1st, 2020. Please upgrade your Python as Python 2.7 is no longer maintained. A future version of pip will drop support for Python 2.7. More details about Python 2 support in pip, can be found at https://pip.pypa.io/en/latest/development/release-process/#python-2-support
You should consider upgrading via the '/usr/local/bin/python -m pip install --upgrade pip' command.
DEPRECATION: Python 2.7 reached the end of its life on January 1st, 2020. Please upgrade your Python as Python 2.7 is no longer main

In [27]:
# Naoqi Related Imports
import naoqi
from naoqi import qi
from naoqi import ALProxy

# Page Related Imports
from pages import displayGeneration

# System Related Imports
import fileinput, os, shutil, sys, random, threading, time

# Flask Related Imports
import json, requests

# Speech Recognition Imports
import pandas as pd
import paramiko
import speech_recognition as sr

## Constants

In [28]:
GPT_HOST = "10.104.23.208"
ip = PEPPER_HOST = "10.104.23.185"

GPT_PORT = 8891
port = PEPPER_PORT = 9559

## Useful Functions
Some functions which can make life easier

### Idle Behaviors

In [29]:
class idling:
    def __init__(self):
        self.idling = True
        self.behavior = ALProxy("ALBehaviorManager", ip, port)
        self.leds = ALProxy("ALLeds", ip, port)
        self.behaviorList = [
            'animations/Stand/Waiting/BackRubs_1', 'animations/Stand/Waiting/Bandmaster_1', 
            'animations/Stand/Waiting/Binoculars_1', 'animations/Stand/Waiting/Drink_1', 
            'animations/Stand/Waiting/Fitness_1', 'animations/Stand/Waiting/Fitness_2', 
            'animations/Stand/Waiting/Fitness_3', 'animations/Stand/Waiting/FunnyDancer_1', 
            'animations/Stand/Waiting/HideEyes_1', 'animations/Stand/Waiting/HideHands_1', 
            'animations/Stand/Waiting/Innocent_1', 'animations/Stand/Waiting/Knight_1',  
            'animations/Stand/Waiting/KungFu_1', 'animations/Stand/Waiting/LookHand_1', 
            'animations/Stand/Waiting/LookHand_2', 'animations/Stand/Waiting/LoveYou_1', 
            'animations/Stand/Waiting/PlayHands_1', 'animations/Stand/Waiting/PlayHands_2', 
            'animations/Stand/Waiting/PlayHands_3', 'animations/Stand/Waiting/Relaxation_1', 
            'animations/Stand/Waiting/Relaxation_2', 'animations/Stand/Waiting/Relaxation_3', 
            'animations/Stand/Waiting/Relaxation_4', 'animations/Stand/Waiting/Rest_1', 
            'animations/Stand/Waiting/ScratchBack_1', 'animations/Stand/Waiting/ScratchBottom_1', 
            'animations/Stand/Waiting/ScratchEye_1',  'animations/Stand/Waiting/ScratchHand_1', 
            'animations/Stand/Waiting/ScratchHead_1', 'animations/Stand/Waiting/ScratchLeg_1', 
            'animations/Stand/Waiting/ScratchTorso_1', 'animations/Stand/Waiting/ShowMuscles_1', 
            'animations/Stand/Waiting/ShowMuscles_2', 'animations/Stand/Waiting/ShowMuscles_3', 
            'animations/Stand/Waiting/ShowMuscles_4', 'animations/Stand/Waiting/ShowMuscles_5', 
            'animations/Stand/Waiting/ShowSky_1', 'animations/Stand/Waiting/ShowSky_2', 
            'animations/Stand/Waiting/Stretch_1', 'animations/Stand/Waiting/Stretch_2',
            'animations/Stand/Waiting/Think_1', 'animations/Stand/Waiting/Think_2', 
            'animations/Stand/Waiting/Think_3', 'animations/Stand/Waiting/Think_4', 
            'animations/Stand/Waiting/Waddle_1', 'animations/Stand/Waiting/Waddle_2', 
            'animations/Stand/Waiting/Zombie_1'
        ]

    def idle_behavior_thread(self, behaviorList):
        while self.idling:
            behaviorName = random.choice(self.behaviorList)

            # Checks if behavior is installed and not running
            if self.behavior.isBehaviorInstalled(behaviorName):
                # Runs behavior
                self.behavior.runBehavior(behaviorName)
                self.leds.fadeRGB('FaceLeds', 'white', 1)
                time.sleep(30)

            else:
                print("Behavior not found: " + behaviorName)

    def start_idle_behavior(self):
        self.idling = True
        self.idle_thread = threading.Thread(target=self.idle_behavior_thread, args=(self.behaviorList,))
        self.idle_thread.start()

    def stop(self):
        self.behavior.stopAllBehaviors()
        self.leds.fadeRGB('FaceLeds', 'white', 1)
        self.idle_thread.join(0)

### Transfer File
Get files from pepper to local machine

In [30]:
def transfer_file(remote_path="/home/nao/microphones/recording.wav", local_path="recordings/recording.wav"):
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    ssh.connect(ip, username='nao', password='nao')

    sftp = ssh.open_sftp()
    sftp.get(remote_path, local_path)

    sftp.close()
    ssh.close()
    print("File transfered")

### Receive File
Send file to pepper

In [31]:
def receive_file(local_path="recordings/recording.wav", remote_path="/home/nao/microphones/recording.wav"):
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    ssh.connect(ip, username='nao', password='nao')

    sftp = ssh.open_sftp()
    sftp.put(local_path, remote_path)

    sftp.close()
    ssh.close()

### Tablet

In [32]:
def show_on_tablet(path):
    if path == 'loading_page':
        path = './pages/dashLoader.html'
    receive_file(path, '/home/nao/.local/share/PackageManager/apps/robot-page/html/page.html')
    tabletService = ALProxy("ALTabletService", ip, port)
    tabletService.loadUrl('http://198.18.0.1/page.html')
    tabletService.showWebview()

In [33]:
def stop_show_on_tablet():
    tabletService = ALProxy("ALTabletService", ip, port)
    tabletService.hideWebview()

In [34]:
stop_show_on_tablet()

In [35]:
displayGeneration.generateBasicQRPage('A221')

copyfile successful
text match found
text match found


### Track Head

In [36]:
def track_head(mode="Head"):
    tracker = ALProxy("ALTracker", ip, port)
    tracker.unregisterAllTargets()
    tracker.initialize()
    tracker.registerTarget("Face", 0.05)
    tracker.setMode(mode)
    tracker.track("Face")

def stop_track_head():
    tracker = ALProxy("ALTracker", ip, port)
    tracker.stopTracker()
    tracker.unregisterAllTargets()

### Change led color

In [37]:
def set_leds(color = 'white', ledSet = 'Face'):
    rgb = None
    if color == 'cyan':
        rgb = [0.69, 1, 0.84]

    leds = ALProxy("ALLeds", ip, port)
    if rgb is not None:   
        leds.fadeRGB(ledSet + 'Leds', rgb[0], rgb[1], rgb[2], 1)
    else:
        leds.fadeRGB(ledSet + 'Leds', color, 1)

In [38]:
set_leds()

### Default Position

In [39]:
def return_to_default_pos(sentences = None):
    posture = ALProxy("ALRobotPosture", ip, port)
    set_leds()
    stop_track_head()
    posture.goToPosture("Stand", 0.5)
    stop_show_on_tablet()

In [73]:
return_to_default_pos()

Exception in thread Thread-63:
Traceback (most recent call last):
  File "/usr/local/lib/python2.7/threading.py", line 801, in __bootstrap_inner
    self.run()
  File "/usr/local/lib/python2.7/threading.py", line 754, in run
    self.__target(*self.__args, **self.__kwargs)
  File "<ipython-input-29-cc274b157783>", line 40, in idle_behavior_thread
    self.behavior.runBehavior(behaviorName)
  File "/pynaoqi-python2.7-2.5.7.1-linux64/lib/python2.7/site-packages/naoqi.py", line 194, in __call__
    return self.__wrapped__.method_missing(self.__method__, *args, **kwargs)
  File "/pynaoqi-python2.7-2.5.7.1-linux64/lib/python2.7/site-packages/naoqi.py", line 264, in method_missing
    raise e
RuntimeError: 	ALBehaviorManager::runBehavior
		ALBehaviorManager::startBehavior
	behavior animations/Stand/Waiting/Innocent_1 is already running.

Exception in thread Thread-62:
Traceback (most recent call last):
  File "/usr/local/lib/python2.7/threading.py", line 801, in __bootstrap_inner
    self.ru

## Detect

In [40]:
# List of greetings
greetings = [
    "Hi! I'm Pepper. How can I assist you?",
    "Hello there! How may I help?",
    "Hello! I'm Pepper, your friendly robot.",
    "Hi there! How can I assist you?",
    "Greetings! What can I do for you today?",
    "Hey! Nice to see you!",
    "Good day! How may I help you?",
    "Welcome! How can I be of service?",
    "Hello! I'm here to make your day better.",
    "Hey! What brings you here?",
    "Hi! How's your day going so far?",
    "Greetings, my friend!",
    "Hello there! I'm always ready to chat.",
    "Hi! I'm Pepper, your helpful companion.",
    "Hey, it's me again! How can I assist you this time?",
    "Good to see you! How can I brighten your day?"
]

In [41]:
def detect():
    # Imports
    memory = ALProxy("ALMemory", ip, port)
    tracker = ALProxy("ALTracker", ip, port)
    face_detection = ALProxy("ALFaceDetection", ip, port)

    face_detection.subscribe('FaceDetection')
    tracker.registerTarget('Face', 0.1)
    tracker.setMode('Head')
    tracker.track('Face')
    tracker.setMaximumDistanceDetection(0.1)

    is_detected = False
    greeting = ""
    while not is_detected:
        faces = memory.getData("FaceDetected")
        if faces and isinstance(faces, list) and len(faces) > 0:
            face_info = faces[0]
            face_id = face_info[0]
            pos = tracker.getTargetPosition(0)
            now = time.time()
            print(pos[0])
            while face_id >= 0 or pos[0] <= 0.85:
                if time.time() - now > 3:
                    is_detected = True
                    # Say a random greeting from the list
                    greeting = random.choice(greetings)
                    break
        say(greeting)

## Listen

### Sound Recording

In [96]:
def timer_callback(timer_cb):
    timer_cb.set()

def record_audio_sd(timer=None, path_name="/home/nao/microphones/recording.wav", debug=False):
    path_name = os.path.join(path_name)

    # Create Connection Proxies
    recorder = ALProxy("ALAudioRecorder", ip, port)
    sound_detector = ALProxy("ALSoundDetection", ip, port)
    memory = ALProxy("ALMemory", ip, port)
    
    sound_detector.subscribe("sound_detector")
    sound_detector.setParameter("Sensitivity", 0.9)

    # Callback function for timer
    timer_cb = threading.Event()

    # Record audio when sound is detected
    if timer is not None:
        threading.Timer(timer, timer_callback, [timer_cb]).start()

    # Start recording
    while True:
        time.sleep(0.5)
        status = memory.getData("SoundDetected")

        # Print status
        if debug:
            print(status)

        # Start recording when sound is detected
        if status is not None:
            print("Recording is starting...")
            recorder.startMicrophonesRecording(path_name, "wav", 16000, (0,0,1,0))
            break

        # Stop recording when timer is reached
        if timer_cb.is_set():
            break

    if timer_cb.is_set():
        recorder.stopMicrophonesRecording()
        sound_detector.unsubscribe("sound_detector")
        return
    
    sound_detector.setParameter("Sensitivity", 0.9)
    while True:
        time.sleep(0.5)
        status = memory.getData("SoundDetected")

        if debug:
            print("Status: ", status)

        # Stop recording when sound stops being detected
        if status is None:
            print("Recording is stopping...")
            recorder.stopMicrophonesRecording()
            sound_detector.unsubscribe("sound_detector")
            break

        # Stop recording when timer is reached
        if timer_cb.is_set():
            print('Timer is reached')
            print("Recording is stopping...")
            recorder.stopMicrophonesRecording()
            sound_detector.unsubscribe("sound_detector")
            break

In [100]:
recorder = ALProxy("ALAudioRecorder", ip, port)

recorder.stopMicrophonesRecording()

In [91]:
transfer_file()

File transfered


In [89]:
timer_cb = threading.Event()
timer_cb.clear()
timer_cb

### Noise Reduction

In [43]:
def reduce_noise(server, amount = 0.5):
    requests.post(server, json={'prop_decrease': amount})

### Speech to Text

In [44]:
def convert_wav_to_text(audio_path):
    r = sr.Recognizer()
    with sr.AudioFile(audio_path) as source:
        audio_data = r.record(source)
    try:
        text = r.recognize_google(audio_data)
        return text
    except sr.UnknownValueError:
        return "Speech recognition could not understand audio"
    except sr.RequestError as e:
        return "Could not request results from Google Speech Recognition service: " + str(e)
    except Exception as e:
        return "An error occurred during speech recognition: " + str(e)

### Listen Function

In [45]:
def listen():
    record_audio_sd(timer=15, debug=False)
    transfer_file()
    reduce_noise()
    text = convert_wav_to_text('recordings/rn_recording.wav')
    return text

## Speak

### Say sentence

In [46]:
def say(data):
    tts = ALProxy("ALTextToSpeech", ip, port)
    tts.say(data)

### Say sentences returned from GPT as they come

In [47]:
global c_code

def say_sentences_thread(sentences):
    # df = pd.read_csv('pages/textbyID.csv')
    # courseID = df['ID'].tolist()
    # check_ID = True
    while True:
        if len(sentences) != 0:

            # if check_ID:
            #     check_ID = False
            #     for word in courseID:
            #         if word in sentences[0]:
            #             c_code = str(sentences[0].split(',')[0])
            #             print('Course Code: ' + c_code)

            if sentences[0] == "quit":
                break  
             
            say(sentences[0])
            sentences.pop(0)

        else:
            time.sleep(1)
            continue
    print('Stopped')

In [48]:
def stop():
    tts = ALProxy("ALTextToSpeech", ip, port)
    tts.stopAll()

## Think

In [49]:
def receive_responses(response, sentences):
    say_thread = threading.Thread(target=say_sentences_thread, args=(sentences,))
    say_thread.start()
    for line in response.iter_content(chunk_size=None):
        x = str(line.decode('utf-8'))
        print(x)
        sentences.append(x)
    say_thread.join()

In [50]:
def think(_question, sentences):
    url = 'http://10.104.22.24:8891/courseInfo'
    # url = "http://{}:{}/courseInfo".format(GPT_HOST, GPT_PORT)
    data = {"question": _question}
    r = requests.post(url, json=data, stream=True)
    receive_responses(r, sentences)
    return

In [51]:
class think_eyes:
    def __init__(self):
        self.thinking = True
        self.leds = ALProxy("ALLeds", ip, port)

    def thinking_eyes(self):
        while self.thinking:
            self.leds.fadeRGB("FaceLeds", 'white', 1)
            time.sleep(1)
            self.leds.fadeRGB("FaceLeds", 0.6, 1, 0.79, 1)

    def start_thinking(self):
        self.eye_thinking_thread = threading.Thread(target=self.thinking_eyes)
        self.eye_thinking_thread.start()

    def stop_thinking(self):
        self.thinking = False
        self.leds.fadeRGB("FaceLeds", 'white', 1)
        self.eye_thinking_thread.join(0)

In [52]:
response = []

think('What are the courses available in Arts because my dad didnt let me pursue my arts degree?', response)

ConnectionError: ('Connection aborted.', BadStatusLine('No status line received - the server has closed the connection',))

## MAIN

In [54]:
idle = idling()

try:
    track_head()
    sentences = []

    while True:
        # translator object initialized during function declaration
        
        # Idle behaviors to keep peppers temperature low
        idle.start_idle_behavior()

        # trigger smart listen
        detect()
        
        time.sleep(5)
        ## If speech detected record up to 15 seconds of audio
        
        record_audio_sd(timer=15, debug=False)
        
        # transfer file from pepper to this machine
        transfer_file()

        # pass audio to speech to text and get text back
        # reduce_noise()
        text = convert_wav_to_text("recordings/recording.wav")
        print(text)
        # print(text, sentences)

        # get gpt response
        say("I'm thinking")
        eyes = think_eyes()
        eyes.start_thinking()
        show_on_tablet('pageTemplates/dashLoader.html')
        
        think(text, sentences)
        eyes.stop_thinking()
        set_leds()
        stop_show_on_tablet()
    
except KeyboardInterrupt:
    recorder = ALProxy("ALAudioRecorder", ip, port)
    recorder.stopMicrophonesRecording()
    return_to_default_pos()
    idle.stop()

finally:
    recorder = ALProxy("ALAudioRecorder", ip, port)
    recorder.stopMicrophonesRecording()
    return_to_default_pos()
    idle.stop()

0.999080777168
Recording is starting...
Recording is stopping...
File transfered
Speech recognition could not understand audio


Exception in thread Thread-21:
Traceback (most recent call last):
  File "/usr/local/lib/python2.7/threading.py", line 801, in __bootstrap_inner
    self.run()
  File "/usr/local/lib/python2.7/threading.py", line 754, in run
    self.__target(*self.__args, **self.__kwargs)
  File "<ipython-input-4-cc274b157783>", line 40, in idle_behavior_thread
    self.behavior.runBehavior(behaviorName)
  File "/pynaoqi-python2.7-2.5.7.1-linux64/lib/python2.7/site-packages/naoqi.py", line 194, in __call__
    return self.__wrapped__.method_missing(self.__method__, *args, **kwargs)
  File "/pynaoqi-python2.7-2.5.7.1-linux64/lib/python2.7/site-packages/naoqi.py", line 264, in method_missing
    raise e
RuntimeError: 	ALBehaviorManager::runBehavior
		ALBehaviorManager::startBehavior
	behavior animations/Stand/Waiting/Relaxation_4 is already running.



- Master of Teaching English (to Speaker's) - This course focuses on research in areas such as principles, approaches, structure analysis features written texts, pedagogical aspects, linguistic/pedagogic factors affecting learners' acquisition and contemporary issues related to curriculum development with a focus on structural barriers new speakers face when acquiring the language

- Diploma of Spanish The Deakin - This program teaches communication skills through grammar, vocabulary building, sentence structures in Spain or other countries where it may be spoken by millions worldwide (with over 400 million speakers) across five continents and offers insights into culture and ways of life within these regions while providing opportunities for career advancement globally as an international language used across multiple cultures

- Diploma of Indonesian The Deakin - This program is designed to develop skills in speaking, writing comprehension, vocabulary building grammar rules sentence 

Exception in thread Thread-12:
Traceback (most recent call last):
  File "/usr/local/lib/python2.7/threading.py", line 801, in __bootstrap_inner
    self.run()
  File "/usr/local/lib/python2.7/threading.py", line 754, in run
    self.__target(*self.__args, **self.__kwargs)
  File "<ipython-input-4-cc274b157783>", line 40, in idle_behavior_thread
    self.behavior.runBehavior(behaviorName)
  File "/pynaoqi-python2.7-2.5.7.1-linux64/lib/python2.7/site-packages/naoqi.py", line 194, in __call__
    return self.__wrapped__.method_missing(self.__method__, *args, **kwargs)
  File "/pynaoqi-python2.7-2.5.7.1-linux64/lib/python2.7/site-packages/naoqi.py", line 264, in method_missing
    raise e
RuntimeError: 	ALBehaviorManager::runBehavior
		ALBehaviorManager::startBehavior
	behavior animations/Stand/Waiting/Fitness_1 is already running.

Exception in thread Thread-35:
Traceback (most recent call last):
  File "/usr/local/lib/python2.7/threading.py", line 801, in __bootstrap_inner
    self.run(

In [ ]:
detect()

0.855526864529


In [ ]:
idle.stop()

In [ ]:
idle.stop()

In [ ]:
memory = ALProxy("ALMemory", ip, port)
gaze = ALProxy('ALGazeAnalysis', ip, port)
gaze.setTolerance(0.1)

for i in range(10):
    time.sleep(3)
    person_id = memory.getData('PeoplePerception/PeopleList')
    if len(person_id) != 0:
        print(memory.getData('GazeAnalysis/PeopleLookingAtRobot'))

None
None
None
None
None
None
None
None


In [ ]:
def receive_responses(response):
    for line in response.iter_content(chunk_size=None):
        print(line.decode())

def send_requests():
    response = requests.post("http://10.104.22.24:8891/courseInfo", json={"question": "How to make cake?"}, stream=True)
    res = receive_responses(response)

send_requests()

Our academic course catalog offers a wide range of courses that cover various disciplines, including science, technology, engineering, arts, and mathematics. From advanced quantum physics to creative writing workshops, we strive to provide diverse learning opportunities for students.
Exploring the world of academia, you'll find our courses designed to ignite intellectual curiosity. Dive deep into the intricacies of ancient civilizations or navigate the complexities of modern data science. Each course is a journey, an exploration of knowledge waiting to be uncovered.
In the realm of academic excellence, our courses stand as pillars of wisdom. Whether you're drawn to unraveling the mysteries of the human mind through psychology courses or delving into the frontiers of artificial intelligence, our curriculum caters to your thirst for learning.
Welcome to our academic universe where courses serve as constellations of knowledge. Embark on a voyage through history, mathematics, literature, a